# Creating a DB in sqlite3


## SQL in sqlite3

Using Jupyter magics

In [1]:
# Capture without a variable sends stdout and stderr to /dev/null
%%capture
%%bash
apt-get update
apt-get install -y sqlite3 wamerican-insane


In [2]:
!sqlite3 --help
!sqlite3 --version

Usage: sqlite3 [OPTIONS] [FILENAME [SQL]]
FILENAME is the name of an SQLite database. A new database is created
if the file does not previously exist. Defaults to :memory:.
OPTIONS include:
   --                   treat no subsequent arguments as options
   -A ARGS...           run ".archive ARGS" and exit
   -append              append the database to the end of the file
   -ascii               set output mode to 'ascii'
   -bail                stop after hitting an error
   -batch               force batch I/O
   -box                 set output mode to 'box'
   -column              set output mode to 'column'
   -cmd COMMAND         run "COMMAND" before reading stdin
   -csv                 set output mode to 'csv'
   -deserialize         open the database using sqlite3_deserialize()
   -echo                print inputs before execution
   -init FILENAME       read/process named file
   -[no]header          turn headers on or off
   -help                show this message
   -html    

In [4]:
!ls -l /usr/share/dict/


total 6768
-rw-r--r-- 1 root root 6922426 Jan 20  2022 american-english-insane
-rw-r--r-- 1 root root     199 Sep  7  2023 README.select-wordlist
lrwxrwxrwx 1 root root      30 Nov  9 18:13 words -> /etc/dictionaries-common/words


In [5]:
!wc -l /usr/share/dict/american-english-insane


663473 /usr/share/dict/american-english-insane


In [6]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane | wc -l


17516


In [7]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane |
egrep '^...u.$' | wc -l


704


In [13]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane |
egrep '^..a.y$' |
egrep '^l.a.y$' |
# egrep '^..aly$' |
egrep -v '[udithpcrzfk]'


leany
leavy
loamy


In [14]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane |
egrep '^..a.y$' |
egrep '^..aly$' |
egrep -v '[udithpcrzfk]'


mealy
sealy
swaly
vealy


In [15]:
%%bash
{
  echo "blah blah blah"
  echo
  echo
  echo words{1..7} | tr ' ' ,
  egrep '^[a-z]{5}$' /usr/share/dict/words |
    shuf -n 420 |
    paste -d, - - - - - - |
    cat -n |
    tr -d ' ' |
    tr '\t' ,
} > /tmp/words.5.csv


In [16]:
# Show the first few entries
!head /tmp/words.5.csv | cat -nvet


     1	blah blah blah$
     2	$
     3	$
     4	words1,words2,words3,words4,words5,words6,words7$
     5	1,typto,beeps,lowns,rupee,regar,faggy$
     6	2,speks,hurry,quilt,merks,modes,rigil$
     7	3,spete,peece,famed,toric,sours,tosca$
     8	4,ulnar,gadus,morns,truff,kipps,husos$
     9	5,joual,porny,pulex,upjet,oozoa,ratus$
    10	6,nandu,tendo,lards,store,abkar,resaw$


In [17]:
!ls -la /tmp/words.5.csv


-rw-r--r-- 1 root root 2789 Nov  9 21:28 /tmp/words.5.csv


In [18]:
# Count the number of lines, words, characters
!wc /tmp/words.5.csv


  74   74 2789 /tmp/words.5.csv


In [19]:
# Remove any existing database
!rm -rf words.sql3


In [20]:
# Import CSV into SQLite3
%%script sqlite3 --column --header words.sql3
.mode csv
.import '| grep -A 100000000 ^words /tmp/words.5.csv' words


In [21]:
# Show that the database now exists
!ls -l


total 12
drwxr-xr-x 1 root root 4096 Nov  6 16:29 sample_data
-rw-r--r-- 1 root root 8192 Nov  9 21:33 words.sql3


In [22]:
# Query the table
%%script sqlite3 --column --header words.sql3
select *
from words
limit 10


words1  words2  words3  words4  words5  words6  words7
------  ------  ------  ------  ------  ------  ------
1       typto   beeps   lowns   rupee   regar   faggy 
2       speks   hurry   quilt   merks   modes   rigil 
3       spete   peece   famed   toric   sours   tosca 
4       ulnar   gadus   morns   truff   kipps   husos 
5       joual   porny   pulex   upjet   oozoa   ratus 
6       nandu   tendo   lards   store   abkar   resaw 
7       nould   darer   taino   spelk   krabs   bilos 
8       tamer   model   noule   irred   buteo   diebs 
9       islay   yfere   nobby   kraut   glady   horsy 
10      audio   sough   aitis   chile   feaze   steak 


In [23]:
# View schema
%%script sqlite3 --column --header words.sql3
.schema


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);


In [24]:
# Create a table with correct data types, including a primary key ( unique, indexed column )
%%script sqlite3 --column --header words.sql3
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);


In [25]:
# View schema
%%script sqlite3 --column --header words.sql3
.schema


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);


In [26]:
# Select data into correct table, i.e. "copy" the data
%%script sqlite3 --column --header words.sql3
INSERT INTO words_real
SELECT *
FROM words


In [28]:
# Show the schema and the first 10 entries
%%script sqlite3 --column --header words.sql3
.schema
select * from words limit 10 ;
select * from words_real limit 10 ;


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
words1  words2  words3  words4  words5  words6  words7
------  ------  ------  ------  ------  ------  ------
1       typto   beeps   lowns   rupee   regar   faggy 
2       speks   hurry   quilt   merks   modes   rigil 
3       spete   peece   famed   toric   sours   tosca 
4       ulnar   gadus   morns   truff   kipps   husos 
5       joual   porny   pulex   upjet   oozoa   ratus 
6       nandu   tendo   lards   store   abkar   resaw 
7       nould   darer   taino   spelk   krabs   bilos 
8       tamer   model   noule   irred   buteo   diebs 
9       islay   yfere   nobby   kraut   glady   horsy 
10      audio   sough   aitis   chile   feaze   steak 
words1  words2  words3  words4  

In [29]:
# Create an index on the last column
%%script sqlite3 --column --header words.sql3
create index if not exists words7 on words_real (words7) ;


In [30]:
# Show the schema, noting the index
%%script sqlite3 --column --header words.sql3
.schema


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
CREATE INDEX words7 on words_real (words7) ;


In [35]:
# Create a derived column
%%script sqlite3 --column --header words.sql3
select *, words6 || "+" || words7 as "both"
from words_real
limit 10


words1  words2  words3  words4  words5  words6  words7  both       
------  ------  ------  ------  ------  ------  ------  -----------
1       typto   beeps   lowns   rupee   regar   faggy   regar+faggy
2       speks   hurry   quilt   merks   modes   rigil   modes+rigil
3       spete   peece   famed   toric   sours   tosca   sours+tosca
4       ulnar   gadus   morns   truff   kipps   husos   kipps+husos
5       joual   porny   pulex   upjet   oozoa   ratus   oozoa+ratus
6       nandu   tendo   lards   store   abkar   resaw   abkar+resaw
7       nould   darer   taino   spelk   krabs   bilos   krabs+bilos
8       tamer   model   noule   irred   buteo   diebs   buteo+diebs
9       islay   yfere   nobby   kraut   glady   horsy   glady+horsy
10      audio   sough   aitis   chile   feaze   steak   feaze+steak


## Using pandas and sqlite3 Python modules

In [36]:
import sqlite3 as db
import pandas as pd


#### Using a context manager

The context manager automatically closes the connection when finished using the database.


In [37]:
# Connect to a sqlite database and load a result set into a data frame
with db.connect("words.sql3") as db_con:
  words = pd.read_sql_query("select * from words", db_con)
words


,words1,words2,words3,words4,words5,words6,words7
0,1,typto,beeps,lowns,rupee,regar,faggy
1,2,speks,hurry,quilt,merks,modes,rigil
2,3,spete,peece,famed,toric,sours,tosca
3,4,ulnar,gadus,morns,truff,kipps,husos
4,5,joual,porny,pulex,upjet,oozoa,ratus
...,...,...,...,...,...,...,...
65,66,dioti,foder,julid,earns,nided,denay
66,67,oxter,barde,apian,bason,lowse,siris
67,68,skits,theft,merer,njord,duras,maned
68,69,haida,hiker,bemud,pouts,lymph,owche


#### Without using a context manager



The connection to the database stays open until you explicitly close it.

In [38]:
# Connect to a sqlite database and load a result set into a data frame
db_con = db.connect("words.sql3")
words = pd.read_sql_query("select * from words", db_con)
words


,words1,words2,words3,words4,words5,words6,words7
0,1,typto,beeps,lowns,rupee,regar,faggy
1,2,speks,hurry,quilt,merks,modes,rigil
2,3,spete,peece,famed,toric,sours,tosca
3,4,ulnar,gadus,morns,truff,kipps,husos
4,5,joual,porny,pulex,upjet,oozoa,ratus
...,...,...,...,...,...,...,...
65,66,dioti,foder,julid,earns,nided,denay
66,67,oxter,barde,apian,bason,lowse,siris
67,68,skits,theft,merer,njord,duras,maned
68,69,haida,hiker,bemud,pouts,lymph,owche


In [39]:
# Show some info about the data frame
words.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   words1  70 non-null     object
 1   words2  70 non-null     object
 2   words3  70 non-null     object
 3   words4  70 non-null     object
 4   words5  70 non-null     object
 5   words6  70 non-null     object
 6   words7  70 non-null     object
dtypes: object(7)
memory usage: 4.0+ KB


In [40]:
# Query the database and load the result into a data frame
words_real = pd.read_sql_query("select * from words_real", db_con)
words_real


,words1,words2,words3,words4,words5,words6,words7
0,1,typto,beeps,lowns,rupee,regar,faggy
1,2,speks,hurry,quilt,merks,modes,rigil
2,3,spete,peece,famed,toric,sours,tosca
3,4,ulnar,gadus,morns,truff,kipps,husos
4,5,joual,porny,pulex,upjet,oozoa,ratus
...,...,...,...,...,...,...,...
65,66,dioti,foder,julid,earns,nided,denay
66,67,oxter,barde,apian,bason,lowse,siris
67,68,skits,theft,merer,njord,duras,maned
68,69,haida,hiker,bemud,pouts,lymph,owche


In [41]:
# Show some info about the data frame
words_real.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   words1  70 non-null     int64 
 1   words2  70 non-null     object
 2   words3  70 non-null     object
 3   words4  70 non-null     object
 4   words5  70 non-null     object
 5   words6  70 non-null     object
 6   words7  70 non-null     object
dtypes: int64(1), object(6)
memory usage: 4.0+ KB


In [42]:
# Show the first 10 entries in the new data frame
words_real2 = words_real[:10]
words_real2


,words1,words2,words3,words4,words5,words6,words7
0,1,typto,beeps,lowns,rupee,regar,faggy
1,2,speks,hurry,quilt,merks,modes,rigil
2,3,spete,peece,famed,toric,sours,tosca
3,4,ulnar,gadus,morns,truff,kipps,husos
4,5,joual,porny,pulex,upjet,oozoa,ratus
5,6,nandu,tendo,lards,store,abkar,resaw
6,7,nould,darer,taino,spelk,krabs,bilos
7,8,tamer,model,noule,irred,buteo,diebs
8,9,islay,yfere,nobby,kraut,glady,horsy
9,10,audio,sough,aitis,chile,feaze,steak


In [43]:
# Save the entire dataframe to a new table in the existing database
words_real2.to_sql("words_real2", db_con, if_exists="replace")


10

In [44]:
# Show the schema of the entire database
%%script sqlite3 --column --header words.sql3
.schema


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
CREATE INDEX words7 on words_real (words7) ;
CREATE TABLE IF NOT EXISTS "words_real2" (
"index" INTEGER,
  "words1" INTEGER,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
CREATE INDEX "ix_words_real2_index"ON "words_real2" ("index");


Notice that we didn't close the connection.


In [45]:
# the right thing to do is to close the connection
db_con.close()


In [46]:
!ls -l


total 28
drwxr-xr-x 1 root root  4096 Nov  6 16:29 sample_data
-rw-r--r-- 1 root root 24576 Nov  9 21:51 words.sql3
